In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
data_path = "../data/processed/multimodal_dataset.csv" 
df = pd.read_csv(data_path)
print(df.head())

# Ensure necessary columns exist
assert all(col in df.columns for col in ['text', 'image_name', 'label'])

# Convert labels to int
df['label'] = df['label'].astype(int)

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

In [ ]:
MAX_VOCAB = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['text'])

def preprocess_text(text_series):
    seqs = tokenizer.texts_to_sequences(text_series)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_text = preprocess_text(train_df['text'])
X_val_text = preprocess_text(val_df['text'])
y_train = train_df['label'].values
y_val = val_df['label'].values

In [ ]:
IMG_SIZE = (128, 128)

def load_and_preprocess_image(img_name):
    try:
        img = load_img(img_name, target_size=IMG_SIZE)
        img = img_to_array(img) / 255.0
        return img
    except:
        return np.zeros((*IMG_SIZE, 3))

X_train_img = np.array([load_and_preprocess_image(p) for p in tqdm(train_df['image_name'], desc="Train images")])
X_val_img = np.array([load_and_preprocess_image(p) for p in tqdm(val_df['image_name'], desc="Val images")])

In [ ]:
# --- Text branch ---
text_input = layers.Input(shape=(MAX_LEN,), name="text_input")
x_text = layers.Embedding(MAX_VOCAB, 128)(text_input)
x_text = layers.Bidirectional(layers.LSTM(64))(x_text)
x_text = layers.Dense(128, activation='relu')(x_text)

# --- Image branch ---
img_input = layers.Input(shape=(*IMG_SIZE, 3), name="image_input")
base_cnn = ResNet50(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base_cnn.trainable = False  # keep frozen for CPU-friendly training
x_img = base_cnn(img_input)
x_img = layers.GlobalAveragePooling2D()(x_img)
x_img = layers.Dense(128, activation='relu')(x_img)

# --- Fusion ---
combined = layers.concatenate([x_text, x_img])
x = layers.Dense(128, activation='relu')(combined)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=[text_input, img_input], outputs=output)
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    [X_train_text, X_train_img],
    y_train,
    validation_data=([X_val_text, X_val_img], y_val),
    epochs=5,
    batch_size=8
)

In [ ]:
val_loss, val_acc = model.evaluate([X_val_text, X_val_img], y_val)
print(f"✅ Validation Accuracy: {val_acc:.4f}")

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title("Model Accuracy")
plt.legend()
plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)
model.save("../models/multimodal_model.h5")